# Cassandra IoT Demo Notebook

## 0. Parameters

In [ ]:
from datetime import date

# Demo parameters
DATE = date.today().strftime("%Y-%m-%d")
# SENSOR_UUID = "<SENSOR_UUID>"  # Replace after metadata lookup
# room_A_SENSOR_UUID = "<room_A_SENSOR_UUID>"  # Optional: known anomalous sensor in room_A
# ANCHOR_VECTOR = "[2150.0, 18000.0, 3.0]"   # Replace with a real profile_vector if needed

## 1. Cassandra Main Properties

In [3]:
%%bash
set -e

echo "
--- nodetool status ---"
docker exec cassandra-1 nodetool status

echo "
--- kafka topics ---"
docker exec kafka kafka-topics --bootstrap-server kafka:29092 --list


--- nodetool status ---
Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load        Tokens  Owns (effective)  Host ID                               Rack 
UN  172.18.0.6  635.62 KiB  16      76.0%             876a86f9-615d-42d9-a284-63629391b044  rack1
UN  172.18.0.2  203.95 KiB  16      64.7%             c81310fd-47fb-428b-bc4a-39f0bbf722e3  rack1
UN  172.18.0.5  129.71 KiB  16      59.3%             18c4b868-214f-48b2-8827-14f3a7e50c26  rack1


--- kafka topics ---
sensor_light
sensor_power
sensor_temp_humidity


### 1.1 Cassandra Masterless Property in action in Read Operation

Note: 
* The coordinator here does not change because we use cqlsh always on the same node (cassandra-1 here). Changing it to cassandra-2 would change the coordinator.
* Notice that, when running a query with CONSISTENCY different than ONE, the READ_REQ for the multiple nodes starts at almost the same time, so parallel request on a P2P network.
  * Example:
    * `READ_REQ message received from /172.18.0.7:7000 [Messaging-EventLoop-3-10] | 2026-04-25 07:19:10.833000 | 127.18.0.5`  
    * `READ_REQ message received from /172.18.0.7:7000 [Messaging-EventLoop-3-5] | 2026-04-25 07:19:10.834000 | 127.18.0.2`

In [1]:
%%bash
set -e

docker exec cassandra-1 cqlsh -e "CONSISTENCY ALL; TRACING ON; SELECT * FROM iot_raw.devices_metadata WHERE sensor_type = 'power' limit 1;"

Consistency level set to ALL.
TRACING set to ON

 sensor_type | sensor_id                            | description                       | location_id
-------------+--------------------------------------+-----------------------------------+-------------
       power | 01dd074c-bfb6-5c17-9c61-e8db891d5bdc | power sensor 5 in room_C (normal) |      room_C

(1 rows)

Tracing session: c22dbaa0-40ae-11f1-9756-559648c1ad05

 activity                                                                                                           | timestamp                  | source     | source_elapsed | client
--------------------------------------------------------------------------------------------------------------------+----------------------------+------------+----------------+-----------
                                                                                                 Execute CQL3 query | 2026-04-25 13:57:55.147000 | 172.18.0.3 |              0 | 127.0.0.1
 Parsing SELECT * F

## 2. Hash ring, token distribution, and replication

### 2.1 Same replication strategy, same replication factor

In [2]:
%%bash
set -e

echo "--- nodetool status ---"
docker exec cassandra-1 nodetool status

echo "Owns sums to 200 because of RF=2"

echo "
--- nodetool ring ---"

docker exec cassandra-1 nodetool ring | sort -k1,1

echo "
--- nodetool ring iot_raw ---"
docker exec cassandra-1 nodetool ring iot_raw | sort -k1,1

echo "
--- nodetool ring iot_raw ---"
docker exec cassandra-1 nodetool ring iot_alerts | sort -k1,1

echo "
--- keyspace replication settings ---"
docker exec cassandra-1 cqlsh -e "
SELECT keyspace_name, replication
FROM system_schema.keyspaces
WHERE keyspace_name IN ('iot_raw','iot_alerts','iot_analytics')
"


--- nodetool status ---
Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load        Tokens  Owns (effective)  Host ID                               Rack 
UN  172.18.0.6  94.49 KiB   16      76.0%             0d712553-3b28-41a6-ae2d-59c42068f22d  rack1
UN  172.18.0.3  133.53 KiB  16      64.7%             870d1dad-fea0-431c-aa8b-8401e49b018b  rack1
UN  172.18.0.5  129.79 KiB  16      59.3%             07edb817-035b-423d-a62f-1a9a86b026ab  rack1

Owns sums to 200 because of RF=2

--- nodetool ring ---




  
172.18.0.3       rack1       Up     Normal  133.53 KiB      64.66%              1269218120452176971                         
172.18.0.3       rack1       Up     Normal  133.53 KiB      64.66%              -1637805833242633650                        
172.18.0.3       rack1       Up     Normal  133.53 KiB      64.66%              2344974001677756510                         
172.18.0.3       rack1       Up     Normal  133.53 KiB      64.66%         

### 2.2 Different Replication Strategy and Factor

In [11]:
%%bash
set -e

echo "--- ALTER KEYSPACE iot_alerts ---"
docker exec cassandra-1 cqlsh -e " ALTER KEYSPACE iot_alerts WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 3};"

echo "--- ALTER KEYSPACE iot_raw ---"
docker exec cassandra-1 cqlsh -e " ALTER KEYSPACE iot_raw WITH replication = {'class': 'NetworkTopologyStrategy', 'dc1': 2};"

--- ALTER KEYSPACE iot_alerts ---

Warnings :
When increasing replication factor you need to run a full (-full) repair to distribute the data.

--- ALTER KEYSPACE iot_raw ---


#### The instructions above change the Replication metadata of the keyspace but does not move the data around

In [13]:
%%bash
set -e

docker exec cassandra-1 nodetool repair --full

[2026-04-25 08:23:32,889] Starting repair command #1 (0c21cf90-4080-11f1-87d0-7faae906987a), repairing keyspace system_traces with repair options (parallelism: parallel, primary range: false, incremental: false, job threads: 1, ColumnFamilies: [], dataCenters: [], hosts: [], previewKind: NONE, # of ranges: 32, pull repair: false, force repair: false, optimise streams: false, ignore unreplicated keyspaces: false, repairPaxos: true, paxosOnly: false)
[2026-04-25 08:23:33,056] Repair session 0c2d8f60-4080-11f1-87d0-7faae906987a for range [(2857239198939660437,3413120233856584373], (-1277325188125006372,-682022819164607121], (944052082285251257,1195041510348378906], (7398479045323913165,7632094981185876697], (3413120233856584373,3824603442018195936], (-7198563030025483682,-6946483601063505060], (-3802638050127738948,-3566368109917616916], (-1602159111067492417,-1277325188125006372], (2061290533156948851,2363996433527393285], (-2740285650457387409,-2502027120144553520], (-825611971471553224

#### Result

Notice the question marks on owning percentage for the whole cluster, different replication factor on different keyspaces does not allow cassandra to compute a percentage that is true for each node.

In [14]:
%%bash
set -e

echo "--- nodetool status ---"
docker exec cassandra-1 nodetool status

echo "Owns sums to 200 because of RF=2"

echo "
--- nodetool ring ---"

docker exec cassandra-1 nodetool ring | sort -k1,1

echo "
--- nodetool ring iot_raw ---"
docker exec cassandra-1 nodetool ring iot_raw | sort -k1,1

echo "
--- nodetool ring iot_raw ---"
docker exec cassandra-1 nodetool ring iot_alerts | sort -k1,1

echo "
--- keyspace replication settings ---"
docker exec cassandra-1 cqlsh -e "
SELECT keyspace_name, replication
FROM system_schema.keyspaces
WHERE keyspace_name IN ('iot_raw','iot_alerts','iot_analytics')
"


--- nodetool status ---
Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load      Tokens  Owns  Host ID                               Rack 
UN  172.18.0.7  3.82 MiB  16      ?     992a8879-14ea-44e1-828b-f16934e04f5f  rack1
UN  172.18.0.2  5.62 MiB  16      ?     803e3bbd-a059-4fa9-880b-aa72eb624f09  rack1
UN  172.18.0.5  4.07 MiB  16      ?     73373fef-78c0-48c5-a904-5cad0a041e02  rack1

Note: Non-system keyspaces don't have the same replication settings, effective ownership information is meaningless
Owns sums to 200 because of RF=2

--- nodetool ring ---




172.18.0.2       rack1       Up     Normal  5.62 MiB        ?                   -1277325188125006372                        
172.18.0.2       rack1       Up     Normal  5.62 MiB        ?                   1613719193259068465                         
172.18.0.2       rack1       Up     Normal  5.62 MiB        ?                   -201569306899426833                         
172.18.0.2       r

### 2.3 Trace a real partition-key query

This query uses the human-readable partition `(location_id, date)` on `readings_by_location`.


In [6]:
%%bash 
set -e

docker exec cassandra-1 cqlsh -e "TRACING ON; SELECT token(location_id, date) as hash, location_id, date FROM iot_raw.readings_by_location WHERE location_id = 'room_A' AND date = '2026-04-25' LIMIT 5;"

TRACING set to ON

 hash                 | location_id | date
----------------------+-------------+------------
 -8516457692688846049 |      room_A | 2026-04-25
 -8516457692688846049 |      room_A | 2026-04-25
 -8516457692688846049 |      room_A | 2026-04-25
 -8516457692688846049 |      room_A | 2026-04-25
 -8516457692688846049 |      room_A | 2026-04-25

(5 rows)

Tracing session: e9b067b0-40b0-11f1-9756-559648c1ad05

 activity                                                                                                                                                                                          | timestamp                  | source     | source_elapsed | client
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------+------------+----------------+-----------
                                               

### 2.2 Direct endpoint lookup

Return the nodes responsible for the given partition key (even if not effectively present in the DB, deterministical not lookup).

In [9]:
!docker exec cassandra-1 nodetool getendpoints iot_raw readings_by_location 'room_A:2020-04-24'

172.18.0.3
172.18.0.6


### 2.3 Partition size balance check (`nodetool tablestats`)

This command prints table-level storage statistics, including **minimum** and **maximum partition bytes**.  


In [ ]:
# Flush the memtable to create an SSTable to ensure
# all data is on disk and reflected in tablestats
!docker exec cassandra-1 nodetool flush
!docker exec cassandra-2 nodetool flush
!docker exec cassandra-3 nodetool flush

!docker exec cassandra-1 nodetool tablestats iot_raw.devices_metadata

Total number of tables: 1
----------------
Keyspace: iot_raw
	Read Count: 1
	Read Latency: 0.265 ms
	Write Count: 22364
	Write Latency: 0.03403702378823109 ms
	Pending Flushes: 0
		Table: devices_metadata
		SSTable count: 1
		Old SSTable count: 0
		Max SSTable size: 7.076KiB
		SSTables in each level: [1, 0, 0, 0, 0, 0, 0, 0, 0]
		SSTable bytes in each level: [2030, 0, 0, 0, 0, 0, 0, 0, 0]
		Space used (live): 7246
		Space used (total): 7246
		Space used by snapshots (total): 0
		Off heap memory used (total): 49
		SSTable Compression Ratio: 0.46991
		Number of partitions (estimate): 4
		Memtable cell count: 180
		Memtable data size: 7228
		Memtable off heap memory used: 0
		Memtable switch count: 3
		Speculative retries: 1
		Local read count: 0
		Local read latency: NaN ms
		Local write count: 3630
		Local write latency: 0.009 ms
		Local read/write ratio: 0.00000
		Pending flushes: 0
		Percent repaired: 0.0
		Bytes repaired: 0B
		Bytes unrepaired: 4.219KiB
		Bytes pending repair: 0B
		B

## 3. Elasticity demo: add node 4 and show redistribution


In [17]:
%%bash
set -e

echo "--- BEFORE node 4 ---"
docker exec cassandra-1 nodetool status
docker exec cassandra-1 nodetool ring iot_raw

docker compose up -d cassandra-4

--- BEFORE node 4 ---
Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load        Tokens  Owns (effective)  Host ID                               Rack 
UN  172.18.0.6  249.47 KiB  16      50.7%             876a86f9-615d-42d9-a284-63629391b044  rack1
UN  172.18.0.7  187.25 KiB  16      52.0%             6bf829d5-eb63-4c9e-9060-352f78fe64b6  rack1
UN  172.18.0.2  215.77 KiB  16      48.5%             c81310fd-47fb-428b-bc4a-39f0bbf722e3  rack1
UN  172.18.0.5  212.92 KiB  16      48.8%             18c4b868-214f-48b2-8827-14f3a7e50c26  rack1


Datacenter: dc1
Address          Rack        Status State   Load            Owns                Token                                       
                                                                                8931197532403896265                         
172.18.0.6       rack1       Up     Normal  249.47 KiB      50.68%              -9084535060374987267                        
172.18.0.7       rack1   

In [18]:
%%bash
set -e

echo "--- AFTER node 4 joins ---"
docker exec cassandra-1 nodetool status
docker exec cassandra-1 nodetool ring iot_raw

--- AFTER node 4 joins ---
Datacenter: dc1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address     Load        Tokens  Owns (effective)  Host ID                               Rack 
UN  172.18.0.6  249.47 KiB  16      50.7%             876a86f9-615d-42d9-a284-63629391b044  rack1
UN  172.18.0.7  152.69 KiB  16      52.0%             6bf829d5-eb63-4c9e-9060-352f78fe64b6  rack1
UN  172.18.0.2  196.46 KiB  16      48.5%             c81310fd-47fb-428b-bc4a-39f0bbf722e3  rack1
UN  172.18.0.5  212.92 KiB  16      48.8%             18c4b868-214f-48b2-8827-14f3a7e50c26  rack1


Datacenter: dc1
Address          Rack        Status State   Load            Owns                Token                                       
                                                                                8931197532403896265                         
172.18.0.6       rack1       Up     Normal  249.47 KiB      50.68%              -9084535060374987267                        
172.18.0.7       rac

## 4. Schema verification and Cassandra data model

Goal: verify the real keyspaces and table names before using them live.


In [20]:
%%bash
set -e

docker exec cassandra-1 cqlsh -e "DESCRIBE KEYSPACES;"

echo "--- iot_raw tables ---"
docker exec cassandra-1 cqlsh -e "USE iot_raw; DESCRIBE TABLES;"

echo "--- iot_alerts tables ---"
docker exec cassandra-1 cqlsh -e "USE iot_alerts; DESCRIBE TABLES;"

echo "--- iot_analytics tables ---"
docker exec cassandra-1 cqlsh -e "USE iot_analytics; DESCRIBE TABLES;"

echo "--- system_schema.tables ---"
docker exec cassandra-1 cqlsh -e "SELECT keyspace_name, table_name FROM system_schema.tables WHERE keyspace_name IN ('iot_raw','iot_alerts','iot_analytics');"


iot_alerts     iot_raw  system_auth         system_schema  system_views         
iot_analytics  system   system_distributed  system_traces  system_virtual_schema

--- iot_raw tables ---

devices_metadata  power_by_sensor       temp_humidity_by_sensor
light_by_sensor   readings_by_location

--- iot_alerts tables ---

sensor_alerts

--- iot_analytics tables ---

aggregates_by_type  sensor_aggregates_30s  sensor_behavior_profiles

--- system_schema.tables ---

 keyspace_name | table_name
---------------+--------------------------
    iot_alerts |            sensor_alerts
 iot_analytics |       aggregates_by_type
 iot_analytics |    sensor_aggregates_30s
 iot_analytics | sensor_behavior_profiles
       iot_raw |         devices_metadata
       iot_raw |          light_by_sensor
       iot_raw |          power_by_sensor
       iot_raw |     readings_by_location
       iot_raw |  temp_humidity_by_sensor

(9 rows)


## 5. Query-first modeling through raw tables

In [31]:
%%bash
set -e

echo "--- devices metadata ---"
docker exec cassandra-1 cqlsh -e "SELECT * FROM iot_raw.devices_metadata where sensor_type = 'temp_humidity' LIMIT 10;"

echo "--- raw readings: temp_humidity ---"
docker exec cassandra-1 cqlsh -e "SELECT * FROM iot_raw.temp_humidity_by_sensor WHERE sensor_id = 028d6827-4c8f-59f8-8ce2-a2663387e93d AND date = '2026-04-25' LIMIT 5;"

--- devices metadata ---

 sensor_type   | sensor_id                            | description                                 | location_id
---------------+--------------------------------------+---------------------------------------------+-------------
 temp_humidity | 028d6827-4c8f-59f8-8ce2-a2663387e93d |   temp_humidity sensor 3 in room_C (normal) |      room_C
 temp_humidity | 033d6766-b7b4-5754-9e28-34cccdc78f19 | temp_humidity sensor 2 in room_A (abnormal) |      room_A
 temp_humidity | 066aac75-28a0-5121-9949-1f58b4b7a441 | temp_humidity sensor 1 in room_A (abnormal) |      room_A
 temp_humidity | 20ae2c28-545a-5bc6-aa88-81aa706481d4 | temp_humidity sensor 0 in room_A (abnormal) |      room_A
 temp_humidity | 2fe5dae0-0d16-555d-9b6f-87bafa50eb28 |   temp_humidity sensor 8 in room_B (normal) |      room_B
 temp_humidity | 3576e082-3d54-533b-b7fc-baa31a30e51b |   temp_humidity sensor 0 in room_C (normal) |      room_C
 temp_humidity | 37027af8-06db-5e9b-8a43-5dcb03654338 |   tem

## 5.1 Sparse wide-row demo with `readings_by_location`

In [34]:
from textwrap import dedent
sparse_query = dedent(f'''
SELECT location_id, date, timestamp, sensor_id, sensor_type,
       temperature, humidity, light_level, amperage, voltage, wattage
FROM iot_raw.readings_by_location
WHERE location_id = 'room_A' AND date = '{DATE}'
LIMIT 12;
''').strip()
print(sparse_query)


SELECT location_id, date, timestamp, sensor_id, sensor_type,
       temperature, humidity, light_level, amperage, voltage, wattage
FROM iot_raw.readings_by_location
WHERE location_id = 'room_A' AND date = '2026-04-25'
LIMIT 12;


In [37]:
%%bash
set -e

docker exec cassandra-1 cqlsh -e "SELECT * FROM iot_raw.readings_by_location WHERE location_id = 'room_A' AND date = '2026-04-25' LIMIT 10;"



 location_id | date       | timestamp                       | sensor_id                            | amperage | humidity | light_level | sensor_type   | temperature | voltage   | wattage
-------------+------------+---------------------------------+--------------------------------------+----------+----------+-------------+---------------+-------------+-----------+------------
      room_A | 2026-04-25 | 2026-04-25 14:30:57.000000+0000 | 033d6766-b7b4-5754-9e28-34cccdc78f19 |     null |    66.64 |        null | temp_humidity |       22.96 |      null |       null
      room_A | 2026-04-25 | 2026-04-25 14:30:57.000000+0000 | 066aac75-28a0-5121-9949-1f58b4b7a441 |     null |    62.15 |        null | temp_humidity |       29.25 |      null |       null
      room_A | 2026-04-25 | 2026-04-25 14:30:57.000000+0000 | 114d73a4-8a47-55cf-a1d7-f520682421a2 |     5.42 |     null |        null |         power |        null | 217.32001 | 1177.87451
      room_A | 2026-04-25 | 2026-04-25 14:30:57.000

## 6. Alerts and 30-second analytics

In [ ]:
%%bash
set -e

docker exec cassandra-1 cqlsh -e "SELECT * FROM iot_alerts.sensor_alerts WHERE sensor_id = 033d6766-b7b4-5754-9e28-34cccdc78f19 LIMIT 10;"

docker exec cassandra-1 cqlsh -e "SELECT * FROM iot_analytics.sensor_aggregates_30s WHERE sensor_id = 033d6766-b7b4-5754-9e28-34cccdc78f19 AND date = '2026-04-25' LIMIT 10;"

docker exec cassandra-1 cqlsh -e "SELECT * FROM iot_analytics.aggregates_by_type WHERE sensor_type = 'temp_humidity' AND date = '2026-04-25' LIMIT 10;"


 sensor_id                            | timestamp                       | alert_id                             | alert_message                                                                   | alert_type    | location_id | severity
--------------------------------------+---------------------------------+--------------------------------------+---------------------------------------------------------------------------------+---------------+-------------+----------
 033d6766-b7b4-5754-9e28-34cccdc78f19 | 2026-04-25 14:30:56.000000+0000 | 83633194-ce54-4320-9b90-c9d0fd129726 | Threshold breached: HUMIDITY_HIGH — sensor 033d6766-b7b4-5754-9e28-34cccdc78f19 | HUMIDITY_HIGH |      room_A |   MEDIUM
 033d6766-b7b4-5754-9e28-34cccdc78f19 | 2026-04-25 14:30:50.000000+0000 | c804573b-f98d-4d92-9ac4-6008fd8f2bd2 | Threshold breached: HUMIDITY_HIGH — sensor 033d6766-b7b4-5754-9e28-34cccdc78f19 | HUMIDITY_HIGH |      room_A |   MEDIUM
 033d6766-b7b4-5754-9e28-34cccdc78f19 | 2026-04-25 14:30:47.00

# CONTINUE FROM HERE

Note: Aggiungere SAI a sensor_alerts in modo da fare query per data (in modo da vedere tutte le stanze e non usare partition key location_id) oppure creare nuova tabella?

## 8. Room A anomaly concentration


In [7]:
%%bash 
set -e

docker exec cassandra-1 cqlsh -e "select count(alert_id) as alerts_count_A from iot_alerts.alerts_by_location where location_id = 'room_A' and date = '2026-04-26';"

docker exec cassandra-1 cqlsh -e "select count(alert_id) as alerts_count_B from iot_alerts.alerts_by_location where location_id = 'room_B' and date = '2026-04-26';"


 alerts_count_a
----------------
             91

(1 rows)

 alerts_count_b
----------------
              9

(1 rows)


We cannot filter by SEVERITY right now given that it is not part of the PK. SAI allows us to create efficient secondary index to filter on non partitioning columns, like SEVERITY.

In [8]:
%%bash 
set -e

docker exec cassandra-1 cqlsh -e "CREATE INDEX IF NOT EXISTS alerts_by_location_severity_idx ON iot_alerts.alerts_by_location (severity) USING 'sai';"

In [18]:
%%bash 
set -e

docker exec cassandra-1 cqlsh -e "select * from iot_alerts.alerts_by_location where location_id = 'room_A' and date = '2026-04-26' and severity = 'HIGH' limit 10;"



 location_id | date       | timestamp                       | sensor_id                            | alert_id                             | alert_message                                                                      | alert_type       | sensor_type   | severity
-------------+------------+---------------------------------+--------------------------------------+--------------------------------------+------------------------------------------------------------------------------------+------------------+---------------+----------
      room_A | 2026-04-26 | 2026-04-25 22:05:32.000000+0000 | 2208bf38-7ab8-57e9-81df-19655c9bad52 | 5f8d1c2a-0eeb-45c2-b82f-761955bfa1a7 |     Threshold breached: VOLTAGE_HIGH — sensor 2208bf38-7ab8-57e9-81df-19655c9bad52 |     VOLTAGE_HIGH |         power |     HIGH
      room_A | 2026-04-26 | 2026-04-25 22:05:32.000000+0000 | 4c23115c-37d5-5911-a3e4-d3fb7716d3e3 | d1021af3-8c1c-4382-a931-747206c34abb |     Threshold breached: VOLTAGE_HIGH — sensor 4c231

## 9. Behavior profiles and Cassandra 5 ANN


Investigate using `sensor_behavior_profiles` table

In [22]:
%%bash
set -e

echo "--- inspect available profiles in room_A/power ---"
docker exec cassandra-1 cqlsh -e "
SELECT location_id, sensor_type, sensor_id, last_updated_at,
       profile_size, mean_value, variance_value, spike_count, profile_vector
FROM iot_analytics.sensor_behavior_profiles
WHERE location_id = 'room_A' AND sensor_type = 'power' LIMIT 10"


--- inspect available profiles in room_A/power ---

 location_id | sensor_type | sensor_id                            | last_updated_at                 | profile_size | mean_value | variance_value | spike_count | profile_vector
-------------+-------------+--------------------------------------+---------------------------------+--------------+------------+----------------+-------------+------------------------------
      room_A |       power | 114d73a4-8a47-55cf-a1d7-f520682421a2 | 2026-04-25 22:05:42.000000+0000 |          133 | 1087.13843 |     4.9772e+05 |          10 | [1087.13843, 4.9772e+05, 10]
      room_A |       power | 2208bf38-7ab8-57e9-81df-19655c9bad52 | 2026-04-25 22:05:42.000000+0000 |          133 | 1202.02979 |     5.6644e+05 |          12 | [1202.02979, 5.6644e+05, 12]
      room_A |       power | 4c23115c-37d5-5911-a3e4-d3fb7716d3e3 | 2026-04-25 22:05:42.000000+0000 |          133 | 1098.40283 |     5.1504e+05 |           9 |  [1098.40283, 5.1504e+05, 9]
      room_

Search for similar behavior using ANN on the `[mean, variance, spike]` vector.

In [33]:
%%bash
set -e

docker exec cassandra-1 cqlsh -e "
SELECT sensor_id, profile_vector, mean_value, variance_value, spike_count
FROM iot_analytics.sensor_behavior_profiles
WHERE location_id = 'room_A' AND sensor_type = 'power'"

docker exec cassandra-1 cqlsh -e "
SELECT sensor_id, profile_vector, mean_value, variance_value, spike_count
FROM iot_analytics.sensor_behavior_profiles
WHERE location_id = 'room_A' AND sensor_type = 'power'
ORDER BY profile_vector ANN OF [1202.02979, 5.6644e+05, 12]
LIMIT 5;"

docker exec cassandra-1 cqlsh -e "
SELECT sensor_id, profile_vector, similarity_cosine(profile_vector, [0, 0, 10]) AS similarity
FROM iot_analytics.sensor_behavior_profiles
WHERE location_id = 'room_A' AND sensor_type = 'power'
ORDER BY profile_vector ANN OF [1202.02979, 5.6644e+05, 12]
LIMIT 5"



 sensor_id                            | profile_vector               | mean_value | variance_value | spike_count
--------------------------------------+------------------------------+------------+----------------+-------------
 114d73a4-8a47-55cf-a1d7-f520682421a2 | [1087.13843, 4.9772e+05, 10] | 1087.13843 |     4.9772e+05 |          10
 2208bf38-7ab8-57e9-81df-19655c9bad52 | [1202.02979, 5.6644e+05, 12] | 1202.02979 |     5.6644e+05 |          12
 4c23115c-37d5-5911-a3e4-d3fb7716d3e3 |  [1098.40283, 5.1504e+05, 9] | 1098.40283 |     5.1504e+05 |           9
 50cda1e2-1ad7-52b6-9c29-5225558bfead |   [987.65027, 2.6166e+05, 1] |  987.65027 |     2.6166e+05 |           1
 53e11baa-cf86-5272-a274-c660a4a1f5bb |   [1036.55383, 2.796e+05, 1] | 1036.55383 |      2.796e+05 |           1
 58fade64-8c68-5310-baed-fb66b430c51e |  [1004.70142, 2.3117e+05, 1] | 1004.70142 |     2.3117e+05 |           1
 a99b078f-185b-594e-b66d-1d6c08f46071 |   [948.61902, 2.1542e+05, 0] |  948.61902 |     2.1542